In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

# load data
vix_data = pd.read_csv('vix_data.csv', parse_dates=['Time'], index_col='Time')
news_data = pd.read_csv('news_data.csv', parse_dates=['Time'], index_col='Time')

# compute sentiment scores
news_data['sentiment_score'] = news_data['headline'].apply(compute_sentiment)

# merge data
data = pd.merge(vix_data, news_data[['sentiment_score']], left_index=True, right_index=True, how='left')

# define event windows
event_dates = pd.date_range(start='2021-01-01', end='2022-12-31', freq='3MS')[:-1]
event_windows = [
    pd.Timedelta(minutes=30),
    pd.Timedelta(hours=1),
    pd.Timedelta(hours=2)
]

# initialize results DataFrame
results = pd.DataFrame(columns=['event_date', 'event_window', 'event_return', 'benchmark_return', 'abnormal_return', 't_statistic', 'p_value'])

# loop over events
for i, event_date in enumerate(event_dates):
    # compute average returns during event window
    event_window = event_windows[i % len(event_windows)]
    event_data = data[(data.index >= (event_date - event_window)) & (data.index <= (event_date + event_window))]
    event_return = event_data['Close'].mean()

    # compute benchmark returns during event window
    benchmark_data = data[(data.index >= (event_date - event_window - pd.Timedelta(minutes=30))) & (data.index <= (event_date - event_window))]
    benchmark_return = benchmark_data['Close'].mean()

    # compute abnormal return
    abnormal_return = event_return - benchmark_return

    # compute standard deviation of benchmark returns
    benchmark_std = benchmark_data['Close'].std()

    # compute t-statistic
    t_statistic = abnormal_return / (benchmark_std / np.sqrt(len(benchmark_data)))

    # compute p-value
    p_value = 2 * (1 - norm.cdf(abs(t_statistic)))

    # add results to DataFrame
    results.loc[i] = [event_date, event_window, event_return, benchmark_return, abnormal_return, t_statistic, p_value]

# plot abnormal returns
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results['event_date'], results['abnormal_return'], marker='o', linestyle='', color='blue')
ax.axhline(y=0, color='black', linestyle='--')
ax.set_title('Abnormal Returns on VIX Based on News Sentiment')
ax.set_xlabel('Event Date')
ax.set_ylabel('Abnormal Return')
ax.xaxis.set_tick_params(rotation=45)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Load the price data for the securities
prices = pd.read_csv('prices.csv', index_col=0, parse_dates=True)

# Load the factor data
factors = pd.read_csv('factors.csv', index_col=0, parse_dates=True)

# Define the factors to be used in the analysis
factors = factors[['Mkt-RF', 'SMB', 'HML', 'RMW']]

# Calculate the excess returns for each security
returns = prices.pct_change().dropna()
rf = factors['RF']
excess_returns = returns.subtract(rf, axis=0)

# Define the event period
event_start = pd.Timestamp('2022-01-01 09:30:00')
event_end = pd.Timestamp('2022-01-01 16:00:00')

# Subset the excess returns to the event period
event_returns = excess_returns.loc[event_start:event_end]

# Calculate the mean excess returns for each security during the event period
mean_returns = event_returns.mean()

# Run the Fama-French four-factor regression for each security
results = []
for ticker in mean_returns.index:
    security_returns = excess_returns[ticker]
    data = pd.concat([security_returns, factors], axis=1).dropna()
    model = sm.OLS(data.iloc[:, 0], sm.add_constant(data.iloc[:, 1:]))
    result = model.fit()
    results.append(result.params)

# Create a DataFrame of the regression coefficients
coefficients = pd.DataFrame(results, index=mean_returns.index, columns=['alpha', 'beta_mkt', 'beta_smb', 'beta_hml', 'beta_rmw'])

# Calculate the abnormal returns for each security
abnormal_returns = pd.DataFrame(index=event_returns.index, columns=event_returns.columns)
for ticker in mean_returns.index:
    expected_returns = coefficients.loc[ticker, 'alpha'] + factors.dot(coefficients.loc[ticker, 'beta_mkt':])
    abnormal_returns[ticker] = event_returns[ticker] - expected_returns

# Calculate the mean abnormal returns during the event period
mean_abnormal_returns = abnormal_returns.mean()

# Plot the mean abnormal returns for each security
fig, ax = plt.subplots(figsize=(10, 6))
mean_abnormal_returns.plot(kind='bar', ax=ax)
ax.axhline(y=0, color='black')
ax.set_title('Mean Abnormal Returns During Event Period')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Load the price data for the securities
prices = pd.read_csv('prices.csv', index_col=0, parse_dates=True)

# Load the factor data
factors = pd.read_csv('factors.csv', index_col=0, parse_dates=True)

# Define the factors to be used in the analysis
factors = factors[['Mkt-RF', 'SMB', 'HML', 'RMW']]

# Define the event periods
event_periods = [
    (pd.Timestamp('2022-01-01 09:30:00'), pd.Timestamp('2022-01-01 16:00:00')),
    (pd.Timestamp('2022-02-01 09:30:00'), pd.Timestamp('2022-02-01 16:00:00')),
    (pd.Timestamp('2022-03-01 09:30:00'), pd.Timestamp('2022-03-01 16:00:00'))
]

# Create an empty DataFrame to store the mean abnormal returns for each event period
mean_abnormal_returns = pd.DataFrame(index=factors.index, columns=[f'event{i+1}' for i in range(len(event_periods))])

# Loop through each event period and calculate the mean abnormal returns for each security
for i, event_period in enumerate(event_periods):
    # Subset the excess returns to the event period
    event_returns = prices.pct_change().subtract(factors['RF'], axis=0).loc[event_period[0]:event_period[1]]
    # Calculate the mean excess returns for each security during the event period
    mean_returns = event_returns.mean()
    # Run the Fama-French four-factor regression for each security
    results = []
    for ticker in mean_returns.index:
        security_returns = event_returns[ticker]
        data = pd.concat([security_returns, factors], axis=1).dropna()
        model = sm.OLS(data.iloc[:, 0], sm.add_constant(data.iloc[:, 1:]))
        result = model.fit()
        results.append(result.params)
    # Create a DataFrame of the regression coefficients
    coefficients = pd.DataFrame(results, index=mean_returns.index, columns=['alpha', 'beta_mkt', 'beta_smb', 'beta_hml', 'beta_rmw'])
    # Calculate the abnormal returns for each security
    abnormal_returns = pd.DataFrame(index=event_returns.index, columns=event_returns.columns)
    for ticker in mean_returns.index:
        expected_returns = coefficients.loc[ticker, 'alpha'] + factors.dot(coefficients.loc[ticker, 'beta_mkt':])
        abnormal_returns[ticker] = event_returns[ticker] - expected_returns
    # Calculate the mean abnormal returns during the event period and store in the DataFrame
    mean_abnormal_returns[f'event{i+1}'] = abnormal_returns.mean()

# Plot the mean abnormal returns for each event period
fig, ax = plt.subplots(figsize=(10, 6))
mean_abnormal_returns.plot(kind='bar', ax=ax)
ax.axhline(y=0, color='black')
ax.set_title('Mean Abnormal Returns During Event Periods')
plt.show()
